# Enlace del histórico con Georgia Tech

Parte de la issue #33 que queda después del módulo de enlace: las tres tasas, la tabla de calibración de Biomédica y un vistazo de 10 enlaces. Lee `historical_matches` ya enlazada y `old_to_new`. No vuelve a enlazar.

Para regenerar el enlace y este informe:

```bash
python -m src.loaders.link_catalog
python -m src.loaders.link_report
```

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.loaders.link_report import (
    INFORME_PATH,
    asignaturas_uc3m,
    cargar_filas,
    es_piloto,
    escribir_informe,
    fuera_de_catalogo,
    grados_con_umbral_propio,
    muestra_enlaces,
    tabla_calibracion,
    tasa,
    tasas_por_grado,
    texto_tasa,
)

filas = cargar_filas()
piloto = tuple(fila for fila in filas if es_piloto(fila))
tasa_piloto = tasa(piloto, "Ingeniería Biomédica")
tasa_todo = tasa(filas, "Georgia Tech")
por_grado = tasas_por_grado(filas)
fuera, no_enlazadas = fuera_de_catalogo(filas)
escribir_informe(filas)

print(f"Biomédica: {texto_tasa(tasa_piloto)}")
print(f"Georgia Tech: {texto_tasa(tasa_todo)}")
print(
    "Grados con al menos 20 pares enlazados: "
    f"{grados_con_umbral_propio(por_grado)} de {len(por_grado)}"
)
print(f"No enlazadas con prefijo fuera del catálogo: {fuera} de {no_enlazadas}")
print(f"Informe: {INFORME_PATH}")

Biomédica: 39/39 (100.0%)
Georgia Tech: 239/244 (98.0%)
Grados con al menos 20 pares enlazados: 4 de 13
No enlazadas con prefijo fuera del catálogo: 0 de 5
Informe: docs\tasas_enlace.md


## Lectura de las tasas

Calculadas con el catálogo de la release `uc3m-2026-10-28-semana3` (2.414 asignaturas de Georgia Tech, 18 departamentos).

Biomédica queda en 39/39 después de quitar los pares repetidos entre cursos. El 49/50 del plan es el recuento anterior a esa deduplicación; el que faltaba por código, `BIOS 7210`, entra ahora por alias como `BIOL 7210`. Está por encima del 50 %, así que la semana 5 puede calibrar sin corregir el enlace antes.

Georgia Tech queda en 239/244 (98,0 %), frente al 263/276 del plan: el denominador ya es la tabla deduplicada (244 filas, no 276). Enlazan 231 por código, 4 por alias y 4 por nombre. Las 5 que no enlazan (`CS 4320`, `CS 4560` ×2, `PHYS 3211` «Electronics 1» y `PHYS 3225`) son de departamentos cargados, pero su código ya no está en el catálogo 2026-27 y ningún nombre se parece; están en `docs/enlace_pendiente.md`.

Por grado, 4 de 13 llegan a 20 pares enlazados, el mínimo de un umbral propio en la fase 2: el doble grado Física + Industriales (42), Biomédica (39), Física (34) e Informática (20). Aeroespacial (16), Robótica (14) e Industriales (14) se quedan cerca.

In [2]:
pd.DataFrame(
    {
        "grado": item.grado,
        "enlazadas": item.enlazadas,
        "total": item.total,
        "tasa": texto_tasa(item),
        "umbral propio": "sí" if item.umbral_propio else "no",
    }
    for item in por_grado
)

,grado,enlazadas,total,tasa,umbral propio
0,Doble Grado en Ingeniería Física e Ingeniería ...,42,42,42/42 (100.0%),sí
1,Grado en Ingeniería Biomédica,39,39,39/39 (100.0%),sí
2,Grado en Ingeniería Física,34,36,34/36 (94.4%),sí
3,Grado en Ingeniería Informática,20,21,20/21 (95.2%),sí
4,Grado en Ingeniería Aeroespacial,16,16,16/16 (100.0%),no
5,Grado en Ingeniería Robótica,14,14,14/14 (100.0%),no
6,Grado en Ingeniería en Tecnologías Industriales,14,14,14/14 (100.0%),no
7,Doble Grado Ciencia e Ingeniería de Datos - In...,12,12,12/12 (100.0%),no
8,Grado en Ingeniería de la Energía,12,12,12/12 (100.0%),no
9,Grado en Matemática Aplicada y Computación,10,11,10/11 (90.9%),no


In [3]:
pares = tabla_calibracion(filas)
print(f"Pares de calibración: {len(pares)}")
print(f"Asignaturas UC3M distintas: {asignaturas_uc3m(pares)}")
pd.DataFrame(
    {
        "código vigente": par.new_code,
        "código antiguo": par.uc3m_codigo,
        "asignatura UC3M": par.uc3m_nombre,
        "reconciliación": par.reconciliation_method,
        "código histórico": par.destino_codigo,
        "nombre histórico": par.destino_nombre,
        "código catálogo": par.catalogo_codigo,
        "nombre catálogo": par.catalogo_nombre,
        "método": par.link_method,
        "grupo": par.grupo,
    }
    for par in pares
)

Pares de calibración: 39
Asignaturas UC3M distintas: 18


,código vigente,código antiguo,asignatura UC3M,reconciliación,código histórico,nombre histórico,código catálogo,nombre catálogo,método,grupo
0,14156,14156,Fisiología Médica I,code,BIOS3753,Human Anatomy,BIOS 3753,Fundamentals of Human Anatomy,code,NaN
1,14156,14156,Fisiología Médica I,code,BIOS3755,Human Physiology,BIOS 3755,Human Physiology,code,774.0
2,14156,14156,Fisiología Médica I,code,BIOS3756,Physiology lab,BIOS 3756,Physiology Laboratory,code,774.0
3,14156,14156,Fisiología Médica I,code,BMED3100,Systems Physiology,BMED 3100,Systems Physiology,code,NaN
4,14157,14157,Fisiología Médica II,code,BIOS3755,Human Physiology,BIOS 3755,Human Physiology,code,NaN
5,14157,14157,Fisiología Médica II,code,BIOS3756,Physiology lab,BIOS 3756,Physiology Laboratory,code,NaN
6,14158,14158,Procesamiento de imágenes médicas,code,BMED4783,Introduction to medical image,BMED 4783,Introduction to Medical Image Processing,code,776.0
7,14158,14158,Procesamiento de imágenes médicas,code,BMED6780,Medical image processing,BMED 6780,Medical Image Processing,code,776.0
8,14159,14159,Biología Computacional,code,BIOS7210,Computational Genomics,BIOL 7210,Computational Genomics,alias,777.0
9,14159,14159,Biología Computacional,code,BIOL7210,Computational Genomics,BIOL 7210,Computational Genomics,code,777.0


Las 39 filas de Biomédica están reconciliadas y enlazadas, así que las 39 entran en la tabla. Son 18 asignaturas UC3M: varias se aprobaron contra más de un curso de Georgia Tech (los grupos n:m y equivalencias distintas). La semana 5 parte el train y el hold-out por asignatura UC3M, no por fila.

En el piloto no hay enlaces por nombre. Los pares son `code`, más el alias `BIOS 7210` → `BIOL 7210`.

In [4]:
pd.DataFrame(
    {
        "id": fila.id,
        "grado": fila.grado,
        "código histórico": fila.destino_codigo,
        "nombre histórico": fila.destino_nombre,
        "código catálogo": fila.catalogo_codigo,
        "nombre catálogo": fila.catalogo_nombre,
        "método": fila.link_method,
    }
    for fila in muestra_enlaces(filas)
)

,id,grado,código histórico,nombre histórico,código catálogo,nombre catálogo,método
0,6533,Grado en Ingeniería Biomédica,BIOS7210,Computational Genomics,BIOL 7210,Computational Genomics,alias
1,2662,Grado en Ingeniería Aeroespacial,AE4699ROM,Undergraduate Research,AE 4699,Undergraduate Research,alias
2,6544,Grado en Ingeniería Física,PHYS3211,Electronics I,PHYS 3209,Electronics I,name
3,9529,Grado en Ingeniería Física,BMED3250,Biomedical systems and modeling,BMED 3520,Biomedical Systems and Modeling,name
4,2677,Grado en Ingeniería Biomédica,BMED3100,Systems Physiology,BMED 3100,Systems Physiology,code
5,2678,Grado en Ingeniería Biomédica,BIOS3755,Human Physiology,BIOS 3755,Human Physiology,code
6,2679,Grado en Ingeniería Biomédica,BMED3211,Introduction to Bioinformatics,BMED 3211,Introduction to Bioinformatics,code
7,2680,Grado en Ingeniería Biomédica,MSE2001,Prin&Appl-Engr Materials,MSE 2001,Principles and Applications of Engineering Mat...,code
8,2681,Grado en Ingeniería Biomédica,BMED3310,Biotransport,BMED 3310,Biotransport,code
9,2682,Grado en Ingeniería Biomédica,BMED4739,Medical Robotics,BMED 4739,Medical Robotics,code


## Vistazo de estos 10

Un enlace repetido en varios grados sale una vez.

- `BIOS 7210` → `BIOL 7210`, alias, en Biomédica. Es la errata conocida y el nombre es el mismo. Correcto.
- `AE 4699 ROM` → `AE 4699`, alias, en Aeroespacial (3 filas). Es el sufijo de Roma; el nombre coincide. Correcto.
- `PHYS 3211` Electronics I → `PHYS 3209` Electronics I, por nombre, en Física y en el doble grado (3 filas). La issue daba `PHYS 3211` como `unlinked` porque no está en el catálogo 2026-27; el título y la descripción de 3209 encajan, así que parece una renumeración. Probable, pendiente de confirmar en la 5A.
- `BMED 3250` → `BMED 3520`, por nombre, en Física. Mismo título y los dígitos invertidos: parece una errata del histórico. La issue también lo daba como `unlinked`. Probable, pendiente de confirmar en la 5A.
- Los seis `code` de Biomédica (`BMED 3100`, `BIOS 3755`, `BMED 3211`, `MSE 2001`, `BMED 3310`, `BMED 4739`) coinciden en código y en nombre.

Ninguno de los enlaces por nombre está en la tabla de calibración: no son de Biomédica. Queda un caso por pulir en el módulo: `PHYS 3211` escrito «Electronics 1» no enlaza porque el ordinal `1` no se compara con `I`. La hoja de las 20 revisiones, `docs/revision_enlace.md`, queda vacía para la semana 5.